# Challenge 06 -- Agent Harness Fundamentals: From Agent to Production Agent

Use this notebook with Student/Challenge-06.md.

Scenario: You are building an AI Workshop Assistant that helps a Microsoft PSA prepare customer workshops.

Learning objective: by the end of this lab, you should be able to explain and demonstrate that:

**Agent = Model + Harness**

## How This Notebook Works

- Python is used throughout.
- Some cells are fully implemented so you can run quickly.
- Some cells include blanks (`___` or `_________`) for you to complete.
- Each section has a reflection prompt. Fill it in before moving on.


## Section 0 -- Introduction

A model can generate text.
An agent can take actions.
An agent harness provides the runtime capabilities that make agents reliable in production.



In [ ]:
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential, ChainedTokenCredential, InteractiveBrowserCredential

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")

openai_client = None
REAL_LLM_READY = False

if PROJECT_ENDPOINT and MODEL_DEPLOYMENT_NAME:
    try:
        credential = ChainedTokenCredential(
            AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
            InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
        )
        project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
        openai_client = project_client.get_openai_client()
        REAL_LLM_READY = True
        print(f"Real LLM ready: {MODEL_DEPLOYMENT_NAME}")
    except Exception as ex:
        print(f"Real LLM setup skipped: {ex}")
else:
    print("Real LLM setup skipped: set AZURE_AI_FOUNDRY_ENDPOINT and AZURE_OPENAI_DEPLOYMENT_NAME in .env")

def fake_chat_completion(prompt: str) -> str:
    # Simulated model-only behavior for lab purposes.
    return f"Model response to: {prompt}"

def real_chat_completion(prompt: str) -> str:
    if not REAL_LLM_READY:
        return "[Real model unavailable in this session.]"
    try:
        response = openai_client.responses.create(
            model=MODEL_DEPLOYMENT_NAME,
            input=prompt,
        )
        return response.output_text
    except Exception as ex:
        return f"[Real model call failed: {ex}]"

def search_workshop_assets(topic: str) -> str:
    assets = {
        "azure ai foundry workshop": "Slides: Foundry-Intro, Demo: Agent Playground, Lab: Tool Calling",
        "ai discovery cards": "Cards: Persona Mapping, Use-Case Prioritization, Value Framing",
        "manufacturing": "Case Study: Predictive Maintenance, Demo: Defect Detection"
    }
    return assets.get(topic.lower(), "No specific assets found.")

def summarize_context(messages):
    if not messages:
        return "Summary: (empty)"
    head = messages[:2]
    tail = messages[-2:] if len(messages) > 2 else []
    combined = head + tail
    return "Summary: " + " | ".join(combined)

def estimate_tokens(text: str) -> int:
    return max(1, len(text.split()))

## Section 1 -- A Model Is Not an Agent

Run the starter code and observe the response.

First run the model-only output, then run the A/B comparison cell to see the difference between model-only and harness-enabled behavior.

Reflection questions:
- Can the model call tools?
- Can the model remember things?
- Can the model execute a workflow?
- Can the model ask for approval?
- Can the model delegate work?

In [ ]:
prompt = "Prepare a customer workshop outline for Contoso."
real_model_only_response = real_chat_completion(prompt)
simulated_model_only_response = fake_chat_completion(prompt)

print("Real model-only response:")
print(real_model_only_response)
print("\nSimulated baseline response:")
print(simulated_model_only_response)

## A/B Comparison -- Model Only vs Harness-Enabled

This comparison uses the same prompt for both paths.

- Model-only path: one LLM call
- Harness-enabled path: plan + tool + memory hint + structured execution summary

In [ ]:
def run_minimal_harness(user_prompt: str):
    plan = [
        "Research customer",
        "Identify AI opportunities",
        "Create workshop agenda",
    ]
    tool_result = search_workshop_assets("Azure AI Foundry workshop")
    memory_hint = "Preferred examples: Manufacturing use cases"
    final_answer = (
        f"Prompt: {user_prompt}\n"
        f"Plan: {plan}\n"
        f"Tool result: {tool_result}\n"
        f"Memory hint: {memory_hint}\n"
        "Execution summary: tool_used=True, approval_required=False"
    )
    return {
        "plan": plan,
        "tool_result": tool_result,
        "memory_hint": memory_hint,
        "final_answer": final_answer,
    }

comparison_prompt = "Prepare a one-day Azure AI Foundry workshop for Fabrikam."
model_only = real_chat_completion(comparison_prompt)
harness_result = run_minimal_harness(comparison_prompt)

print("Model-only output:\n")
print(model_only)
print("\n" + "=" * 60 + "\n")
print("Harness-enabled output:\n")
print(harness_result["final_answer"])

YOUR ANSWER:
___________________________

## Section 2 -- Build Your First Agent Loop

Most production agents use a loop.

Pseudocode:

while not complete:
    ask model what to do
    execute action
    update context
    continue

Student challenge: complete the missing code.

In [ ]:
context = ["User asks for workshop assets on Azure AI Foundry"]

@dataclass
class LoopResponse:
    requires_tool: bool
    tool_name: str
    tool_input: str
    final_answer: str

def agent_step(ctx):
    latest = ctx[-1].lower()
    if "assets" in latest:
        return LoopResponse(True, "search_workshop_assets", "Azure AI Foundry workshop", "")
    return LoopResponse(False, "", "", "Draft complete.")

response = __________

if response.requires_tool:
    tool_result = __________
    context.append(tool_result)

print(context)

Discussion:
- Why is a loop required?
- How is this different from a single model invocation?

Reflection:
What happens if the agent cannot continue after using a tool?

YOUR ANSWER:
___________________________

## Section 3 -- Tools

Introduce a tool:
`search_workshop_assets(topic)`

Student challenge: register the tool with the agent.

In [ ]:
tools = [
    __________
]

test_prompt = "Find assets for an Azure AI Foundry workshop."
tool_output = tools[0]("Azure AI Foundry workshop")
final_answer = f"Prompt: {test_prompt}\nTool output: {tool_output}"

print(final_answer)

Observe:
- tool invocation
- result returned
- final answer

Reflection:
Without tools, what information is the model limited to?

YOUR ANSWER:
___________________________

## Section 4 -- Memory

Agents lose information between sessions unless memory is added.

Starter memory store:

In [ ]:
memory_store = {}

def save_memory(key, value):
    _________

def load_memory(key):
    _________

save_memory("workshop_preferences", "My preferred workshop examples are Manufacturing use cases.")

# Simulate a new run
new_run_question = "What workshop examples do I prefer?"
print(new_run_question)
print(load_memory("workshop_preferences"))

Reflection:
Why is memory a harness capability and not a model capability?

YOUR ANSWER:
___________________________

## Section 5 -- Context Management

Every model has finite context. Implement compaction logic when the context is too long.

In [ ]:
LIMIT = 35
context = [
    "User: Build an executive workshop agenda.",
    "Assistant: Drafted a 1-day agenda skeleton.",
    "User: Add manufacturing examples and architecture options.",
    "Assistant: Added examples, diagrams, and implementation notes.",
    "User: Include follow-up tasks and role assignments."
]

token_count = sum(estimate_tokens(msg) for msg in context)
print(f"Before token count: {token_count}")

if token_count > LIMIT:
    summary = __________
    context = [summary]

after_count = sum(estimate_tokens(msg) for msg in context)
print(f"After token count: {after_count}")
print(context)

Reflection:
Why can long-running agents fail without context management?

YOUR ANSWER:
___________________________

## Section 6 -- Planning

Prompt: "Create an AI Discovery Cards workshop for Contoso."

Without planning: one-shot response
With planning: todo list

Student challenge: complete todo list construction.

In [ ]:
todo_list = []

todo_list.append(
    ___________
)

print("Generated plan:")
for i, item in enumerate(todo_list, start=1):
    print(f"{i}. {item}")

Expected plan should include:
- Research customer
- Identify AI opportunities
- Create agenda
- Generate workshop preparation guide

Reflection:
Why should agents plan before executing?

YOUR ANSWER:
___________________________

## Section 7 -- Sub-agents

Large tasks can be broken into specialized agents.

Architecture:
Coordinator
  |- ResearchAgent
  |- AgendaAgent
  '- SummaryAgent

In [ ]:
def ResearchAgent(customer):
    return f"Research for {customer}: strategic priorities and constraints"

def AgendaAgent(research):
    return f"Agenda built from research: {research}"

def SummaryAgent(research, agenda):
    return f"Summary ready.\n- {research}\n- {agenda}"

customer = "Contoso"
research_results = __________
agenda = __________
final_output = __________

print(final_output)

Reflection:
What advantages do specialized agents provide?

YOUR ANSWER:
___________________________

## Section 8 -- Human in the Loop

Not all actions should be performed automatically.

Scenario: Agent wants to send workshop follow-up emails.

Student challenge: fill in approval logic and test both paths (approved and denied).

In [ ]:
def stop():
    return "Action stopped by approval gate."

action = "send_email"

if action == "send_email":
    approved = __________
    if not approved:
        print(stop())
    else:
        print("Email sent.")

Reflection:
Why is human approval important in enterprise scenarios?

YOUR ANSWER:
___________________________

## Section 9 -- Observability

Production agents must be monitored.

Student challenge: capture and display tool calls, token counts, latency, and run duration.

In [ ]:
events = []

def log_event(name, payload):
    events.append({"event": name, "payload": payload, "timestamp": time.time()})

start_time = time.time()

# TODO: log tool calls
# TODO: log token counts
# TODO: log latency

time.sleep(0.05)
duration = time.time() - start_time

tools_used = __________
tokens_used = __________
estimated_cost = round(tokens_used * 0.000002, 6)

print("Agent Run Summary")
print("-------------------")
print(f"Tools Used: {tools_used}")
print(f"Tokens: {tokens_used}")
print(f"Duration: {duration:.4f} seconds")
print(f"Estimated Cost: ${estimated_cost}")

Reflection:
What problems become difficult to diagnose without observability?

YOUR ANSWER:
___________________________

## Section 10 -- Capstone

Challenge: Build an AI Workshop Assistant Harness.

Requirements:
- Use tools
- Use memory
- Create a plan
- Use sub-agents
- Request approval before sending content
- Capture observability metrics

Input:
"Prepare a one-day Azure AI Foundry workshop for Fabrikam."

Output must contain:
- Research summary
- Workshop agenda
- Recommended demos
- Architecture recommendations
- Follow-up actions
- Execution summary

In [ ]:
# Implement your full harness flow here.
# Suggested approach:
# 1) Create plan
# 2) Call sub-agents
# 3) Persist preference memory
# 4) Ask for approval before final send action
# 5) Emit observability metrics

capstone_input = "Prepare a one-day Azure AI Foundry workshop for Fabrikam."
print(capstone_input)

# YOUR IMPLEMENTATION


## Final Reflection

Complete the table:

| Capability | Model Only | Agent Harness |
|------------|-----------|---------------|
| Tool Use | | |
| Memory | | |
| Planning | | |
| Context Management | | |
| Sub-Agents | | |
| Human Approval | | |
| Observability | | |

Final question:
In your own words, explain why an agent harness is required for production-grade agents.

YOUR ANSWER:
___________________________

## Bonus Challenge

Create a reusable Python class: `AgentHarness`.

Capabilities:
- register_tool()
- save_memory()
- create_plan()
- request_approval()
- run()

Implement missing methods and use the harness to execute the workflow.

In [ ]:
class AgentHarness:
    def __init__(self):
        self.tools = {}
        self.memory = {}

    def register_tool(self, name, fn):
        __________

    def save_memory(self, key, value):
        __________

    def create_plan(self, customer):
        __________

    def request_approval(self, action):
        __________

    def run(self, prompt):
        __________

h = AgentHarness()
h.register_tool("search_workshop_assets", search_workshop_assets)
print("Harness ready. Implement methods, then run capstone through h.run(...).")